# Custom mode

This notebook builds a small custom-mode panel. It keeps a selected headlight color applied and announces the selected mode name through the robot speaker.

Run the cells from top to bottom while the robot is connected to the same DDS network. Keep the announcement short because it is spoken out loud.


Load local modules and choose the DDS interface.


In [1]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Configured for iface='eth0', domain_id=0.


Import the robot wrapper and notebook UI helpers.


In [2]:
import threading
import time

import ipywidgets as widgets
from IPython.display import display

from sdk_client import Robot


Create the robot object. Sensor startup is disabled because this exercise only needs lights and speech.


In [3]:
robot = Robot(
    iface=IFACE,
    domain_id=DOMAIN_ID,
    safety_boot=False,
    recover_dev_mode_on_init=False,
    auto_start_sensors=False,
)
print("Robot client ready.")


Robot client ready.


The controller stores the desired mode name and color, re-applies the headlight periodically, and can announce the mode name on demand.


In [4]:
class CustomModeController:
    def __init__(self, robot, repeat_s=10.0):
        self.robot = robot
        self.repeat_s = float(repeat_s)
        self.mode_name = "academy custom mode"
        self.color = "#1e90ff"
        self.intensity = 80
        self.enabled = False
        self._lock = threading.RLock()
        self._stop = threading.Event()
        self._thread = None

    def start(self):
        with self._lock:
            self.enabled = True
            self._apply_locked()
            if self._thread is None or not self._thread.is_alive():
                self._stop.clear()
                self._thread = threading.Thread(target=self._loop, daemon=True)
                self._thread.start()
        return "Persistent custom mode is running."

    def stop(self):
        with self._lock:
            self.enabled = False
            self._stop.set()
        return "Persistent custom mode stopped; last color remains on the robot."

    def update(self, mode_name=None, color=None, intensity=None, repeat_s=None):
        with self._lock:
            if mode_name is not None:
                self.mode_name = str(mode_name).strip() or self.mode_name
            if color is not None:
                self.color = str(color)
            if intensity is not None:
                self.intensity = max(0, min(100, int(intensity)))
            if repeat_s is not None:
                self.repeat_s = max(1.0, float(repeat_s))
            if self.enabled:
                self._apply_locked()
        return self.summary()

    def announce(self):
        text = f"Custom mode active: {self.mode_name}"
        code = self.robot.say(text)
        return f"Announced: {text!r}; Robot.say returned {code}."

    def summary(self):
        return f"mode={self.mode_name!r} color={self.color} intensity={self.intensity} repeat={self.repeat_s:.1f}s enabled={self.enabled}"

    def _apply_locked(self):
        return self.robot.headlight(color=self.color, intensity=self.intensity, duration=None)

    def _loop(self):
        while not self._stop.wait(self.repeat_s):
            with self._lock:
                if self.enabled:
                    try:
                        self._apply_locked()
                    except Exception:
                        pass

custom_mode = CustomModeController(robot)
print(custom_mode.summary())


mode='academy custom mode' color=#1e90ff intensity=80 repeat=10.0s enabled=False


Run the panel. Use Start to begin periodic color refresh, Announce to speak the current mode, and Stop when the exercise is done.


In [5]:
mode_name = widgets.Text(value="academy custom mode", description="Mode", layout=widgets.Layout(width="420px"))
color = widgets.ColorPicker(value="#1e90ff", description="Color")
intensity = widgets.IntSlider(value=80, min=0, max=100, step=5, description="Intensity")
repeat_s = widgets.FloatSlider(value=10.0, min=1.0, max=60.0, step=1.0, description="Repeat s")
start_button = widgets.Button(description="Start", button_style="success")
announce_button = widgets.Button(description="Announce", button_style="info")
stop_button = widgets.Button(description="Stop", button_style="warning")
status = widgets.HTML(value="")


def sync_settings():
    return custom_mode.update(mode_name.value, color.value, intensity.value, repeat_s.value)


def on_start(_):
    try:
        sync_settings()
        status.value = custom_mode.start()
    except Exception as exc:
        status.value = f"Start failed: {exc}"


def on_announce(_):
    try:
        sync_settings()
        status.value = custom_mode.announce()
    except Exception as exc:
        status.value = f"Announcement failed: {exc}"


def on_stop(_):
    status.value = custom_mode.stop()

for widget in (mode_name, color, intensity, repeat_s):
    widget.observe(lambda _change: setattr(status, "value", sync_settings()), names="value")
start_button.on_click(on_start)
announce_button.on_click(on_announce)
stop_button.on_click(on_stop)
status.value = custom_mode.summary()
display(widgets.VBox([mode_name, widgets.HBox([color, intensity, repeat_s]), widgets.HBox([start_button, announce_button, stop_button]), status]))
